# **DDDM - Mini - Project 2 - Classification: Neighbourhoods Like Ladywood**

In Project 1 you found that Ladywood is unlike the rest of Birmingham. Half of
its households rent privately, more than twice the share across the city, only
about one in six owns their home, and it has the **highest proportion of
car-free households of any of the city's 69 wards**.

That raises a question a designer actually needs answered. If you are proposing
something for Ladywood, where else might it work? And what is it about a
neighbourhood that makes it car-free in the first place?

This project answers both with **k-nearest neighbours**, the last technique in
the course. You will build a classifier that predicts whether a neighbourhood is
car-light from how its residents live, and then use the same machinery to find
the places most similar to Ladywood.

**This project has hidden tests.** Passing every visible test does not guarantee
full marks.

**If a test fails**, check that you spelled the variable name exactly as the
question asked, that you ran the cell, and that every cell above it has been run
from the top. Most failures are one of those three.

> **This is version 2 of the project.** If you still have the earlier copy open, use this one instead: the first cell below installs the libraries, which the earlier copy was missing. Nothing else has changed, so any work you have done can be pasted across.


## **Do not change the code in the two cells below** ##

In [ ]:
# Run this cell first to install the libraries. It may take a minute.
%pip install -q urllib3<2.0 otter-grader==7.0.0 datascience ipywidgets

In [ ]:
# Run this cell to set up the notebook, but please don't change it.

try:
    import pyodide_http
    pyodide_http.patch_all()
except ImportError:
    pass

import numpy as np
from datascience import *

import matplotlib
%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')
import warnings
warnings.simplefilter('ignore', FutureWarning)

import hashlib
import otter
grader = otter.Notebook("project2-v2.ipynb")

# The helper for the 'Just run this cell' cells lives in project_helpers.py,
# which must sit in the same folder as this notebook.
try:
    from project_helpers import needs, summary
except ImportError:
    raise ImportError(
        'project_helpers.py was not found. It must be in the same folder as this '
        'notebook. Download it from Ulwazi alongside the notebook and the CSV '
        'files, then run this cell again.')


---

# **Part 1: The dataset**

Each row is one **Lower Layer Super Output Area** in Birmingham, about 1,500
people. There are 659 of them.

The first five columns are not features. Two of them name the area, `LSOA` and
`Ward`; the other three are `Households`, `No car %` and `Car status`, the last
being the label you will predict. Every column after those five is a
**proportion**, and the proportions come in four blocks: tenure, housing type,
household deprivation, and ethnicity. Within each block the values sum to 1.

That matters. Because every feature is a proportion, they are already on the
same scale, and you can compare neighbourhoods by straightforward distance
without converting anything to standard units first.

In [ ]:
# Just run this cell
lsoas = Table.read_table('birmingham_lsoa.csv')
print(lsoas.num_rows, 'neighbourhoods,', lsoas.num_columns - 5, 'features')
lsoas.select('LSOA', 'Ward', 'Households', 'No car %', 'Car status').show(5)

The label you will predict is `Car status`. A neighbourhood is **car-light** if
more than 35% of its households have no car or van, and **car-dependent**
otherwise. Those two strings, `'car-light'` and `'car-dependent'`, are the exact
values in the column.

---

**Question 1.1.** How balanced are the two classes? **(8 Points)**

Assign:

1. `car_light_proportion` to the proportion of the 659 neighbourhoods that are car-light
2. `baseline_accuracy` to the accuracy you would get from a classifier that ignores the data entirely and always guesses the **more common** class

*The first line is written apart from the two blanks in it:* the name of the column holding the label, and the label itself. Comparing a column to a value gives an array of `True` and `False`, and `np.count_nonzero` counts the `True`s.

*Hint:* one of these two numbers is `1` minus the other. Which way round depends on which class is more common, so work that out before you decide. A classifier that always guesses the more common class can never be right less than half the time.

In [ ]:
car_light_proportion = np.count_nonzero(lsoas.column(...) == ...) / lsoas.num_rows
baseline_accuracy = ...

print('Car-light proportion:', car_light_proportion)
print('Baseline accuracy:   ', baseline_accuracy)

In [ ]:
grader.check("q1_1")

> **Remember `baseline_accuracy`.** That is the score to beat. Whatever your
> classifier achieves has to be measured against a guesser that already knows
> which class is more common, not against 50%.

---

## **Exploring the features**

Before building anything, look at how the features relate to each other. Two
proportions that move together tell you something about how neighbourhoods
work.

---

**Question 1.2.** Compute the correlation between the proportion of households renting from a private landlord and the proportion living in a purpose-built flat.

Write `standard_units` and use it to compute `r_rent_flat`. **(8 Points)**

In [ ]:
def standard_units(arr):
    return (arr - ...) / ...

rent = lsoas.column('tenure: private rented: private landlord or letting agency')
flats = lsoas.column('housing: purpose-built flat')

r_rent_flat = ...
r_rent_flat

In [ ]:
grader.check("q1_2")

In [ ]:
# Just run this cell
with needs('Question 1.2'):
    lsoas.scatter('tenure: private rented: private landlord or letting agency',
                  'housing: purpose-built flat')
    plots.title(f'r = {round(r_rent_flat, 3)}')
    plots.show()


Private renting and purpose-built flats are related, but loosely: the cloud is
wide and plenty of neighbourhoods sit well off the trend. Neither causes the
other, and neither predicts the other reliably on its own. What they do is
describe the same kind of place, and a classifier can use that alongside
everything else.

---

## **Splitting the dataset**

You cannot judge a classifier on data it has already seen. Split the
neighbourhoods into a **training set**, which the classifier learns from, and a
**test set**, which you keep back to measure it.

If a neighbourhood ended up in both sets, the test would be measuring how well
the classifier remembers rather than how well it generalises, and the accuracy
would come out too high.

---

**Question 1.3.** Shuffle the rows, then put the first 150 into `test_lsoas` and the rest into `train_lsoas`. **(10 Points)**

Assign the shuffled table to `shuffled` first, then take slices of it.

*Hint:* `tbl.sample` is the tool, but check its default before you use it. Sampling **with** replacement would give you a table with some rows repeated and others missing, which is not a shuffle. Pass `with_replacement=False`.

*Note:* Do not change the seed line. Everyone needs the same split for the tests to agree.

In [ ]:
np.random.seed(42) # DO NOT CHANGE THIS LINE

shuffled = lsoas.sample(...)
test_lsoas = shuffled.take(np.arange(...))
train_lsoas = shuffled.take(np.arange(..., ...))

print('Training:', train_lsoas.num_rows, ' Test:', test_lsoas.num_rows)

In [ ]:
grader.check("q1_3")

---

# **Part 2: Distance**

To find a neighbourhood's nearest neighbours you need a definition of "near".

For two neighbourhoods described by the same list of proportions, the
**Euclidean distance** is the square root of the sum of the squared differences,
feature by feature. It is the ordinary straight-line distance, in as many
dimensions as you have features.

---

**Question 2.1.** Write a function `distance` that takes two arrays of feature values and returns the Euclidean distance between them. It must work for arrays of any length, as long as both are the same length. **(6 Points)**

In [ ]:
def distance(features1, features2):
    '''The Euclidean distance between two arrays of feature values.'''
    return np.sqrt(sum(...))

# A quick check: the distance from something to itself is zero
distance(make_array(0.1, 0.4), make_array(0.1, 0.4))

In [ ]:
grader.check("q2_1")

The cell below pulls out just the feature columns, dropping those first five, and uses your function to measure how far apart the first two training
neighbourhoods are.

In [ ]:
# Just run this cell
with needs('Questions 1.3 and 2.1'):
    train_features = train_lsoas.drop('LSOA', 'Ward', 'Households', 'No car %', 'Car status')
    test_features = test_lsoas.drop('LSOA', 'Ward', 'Households', 'No car %', 'Car status')

    first_two_distance = distance(np.array(train_features.row(0)),
                                  np.array(train_features.row(1)))
    print('Distance between the first two training neighbourhoods:', first_two_distance)


---

Computing distances one pair at a time is slow. Run the cell below to define
`fast_distances`, which computes the distance from one neighbourhood to every
row of a table at once. You will use it for the rest of the project.

In [ ]:
# Just run this cell
def fast_distances(test_row, train_table):
    '''Distance from test_row to every row of train_table.'''
    train_matrix = np.array(train_table.columns).T
    diff = train_matrix - np.array(list(test_row))
    return np.sqrt((diff ** 2).sum(axis=1))

# It should agree with your own function on a single row
with needs('Questions 1.3 and 2.1', 'fast_distances itself is defined either way.'):
    check = fast_distances(train_features.row(1), train_features.take(0))
    print('fast_distances:', round(check.item(0), 6))
    print('your distance: ', round(first_two_distance, 6))


---

**Question 2.2.** Take the first neighbourhood in the test set. Produce a table called `neighbours` with two columns, `Car status` and `Distance`, holding every training neighbourhood's label and its distance to that test row, **sorted so the closest comes first**. **(10 Points)**

The cell pairs the labels with the distances for you. You supply the three pieces it needs, and the sort.

*Hint:* `fast_distances` wants a row of features and a table of features, not the full tables with their text columns.

*Hint:* `fast_distances` returns the distances in the same order as the training table's rows, which is why the labels can be put beside them directly. Sorting the finished table moves whole rows, so each distance keeps its own label.

In [ ]:
first_test_row = test_features.row(...)
dists  = fast_distances(..., ...)
labels = train_lsoas.column(...)

unsorted = Table().with_columns('Car status', labels, 'Distance', dists)
neighbours = unsorted.sort(...)
neighbours.show(5)

In [ ]:
grader.check("q2_2")

---

# **Part 3: The classifier**

The rule is simple. To classify a neighbourhood, find the k training
neighbourhoods closest to it and take a **majority vote** of their labels.

---

**Question 3.1.** Write `most_common`, which returns the most frequent value in a column of a table. **(6 Points)**

*The grouping is done; the rest is yours.*

*Hint:* look at `counted.labels` to see what `group` called the counting column before you try to sort on it.

In [ ]:
def most_common(label, table):
    '''The most common value in the given column of a table.'''
    counted = table.group(label)
    return counted.sort(..., descending=True).column(...).item(0)

most_common('Car status', neighbours.take(np.arange(11)))

In [ ]:
grader.check("q3_1")

---

**Question 3.2.** Now put it together. Write `classify`, which takes a row of features, the training features, the training labels and a value of k, and returns the predicted class. **(14 Points)**

*You built the same kind of table at Question 2.2.* The first line is complete; the two below it have blanks.

In [ ]:
def classify(test_row, train_features, train_labels, k):
    '''Predict a class by majority vote among the k nearest neighbours.'''
    distances = fast_distances(test_row, train_features)
    nearest = Table().with_columns('Class', train_labels, 'Distance', distances).sort(...).take(np.arange(...))
    return most_common(..., nearest)

classify(test_features.row(0), train_features, train_lsoas.column('Car status'), 11)

In [ ]:
grader.check("q3_2")

---

**Question 3.3.** Classify every neighbourhood in the test set with k = 11, then see whether the classifier is worth anything. **(10 Points)**

Assign:

1. `test_predictions` to the predicted class for every test neighbourhood
2. `test_accuracy` to the proportion your classifier got right
3. `beats_baseline` to `True` if `test_accuracy` is higher than `baseline_accuracy` from **Question 1.1**, and `False` otherwise

*The calls are written for you; you fill in what goes inside them.*

*Hint:* `tbl.apply(function)` calls your function once for each row of a table and passes it only that row. That is why the cell defines `classify_lsoa`: `classify` needs four arguments and `apply` can only supply one, so the wrapper holds the other three. Its two blanks are the labels column and the value of k.

*Hint:* not every blank is an argument. In `test_accuracy` the `/ ...` wants the number of test neighbourhoods, and in `beats_baseline` the `bool(... > ...)` wants the two accuracies being compared.

*If this fails and so did **Question 3.2**, fix that one first.*

In [ ]:
def classify_lsoa(row):
    return classify(row, train_features, train_lsoas.column(...), ...)

test_predictions = test_features.apply(...)
test_accuracy    = np.count_nonzero(test_predictions == test_lsoas.column(...)) / ...
beats_baseline   = bool(... > ...)

print('Accuracy:', test_accuracy, ' Baseline:', baseline_accuracy, ' Beats it:', beats_baseline)

In [ ]:
grader.check("q3_3")

---

**Question 3.4.** Nothing so far has justified k = 11. It was simply handed to you. Find out whether it was a good choice. **(10 Points)**

Assign:

1. `k_accuracies` to an array of five test-set accuracies, one for each value in `k_values`, in the same order
2. `best_k` to whichever value of k gave the highest test accuracy
3. `k_reason` to 1, 2 or 3 below, whichever best explains the pattern you see

The reason:

1. A larger k is always better, because the vote uses more information.
2. With k = 1 a single unusual neighbour decides the answer, and with a very large k the vote is filled out by neighbourhoods that are not similar at all. The useful value sits between the two.
3. The best k is whichever gives the highest accuracy on the training set.

*The loop is written for you.* It runs once per value in `k_values`, and its two blanks are the value of k to classify at and the number of test neighbourhoods to divide by. The lines around it have blanks of their own: `k_accuracies` must start as an empty array for `np.append` to build on, and `k_reason` is one of the three numbers above.

*Hint:* `best_k` is read off a two-column table of `k` against `accuracy`, sorted. The blanks are the column to sort on and the column to read back.

In [ ]:
k_values = make_array(1, 5, 11, 25, 101)

k_accuracies = ...        # an empty array for the five accuracies
for k in k_values:
    preds = test_features.apply(lambda r: classify(r, train_features, train_lsoas.column('Car status'), ...))
    k_accuracies = np.append(k_accuracies, np.count_nonzero(preds == test_lsoas.column('Car status')) / ...)
best_k       = Table().with_columns('k', k_values, 'accuracy', k_accuracies).sort(..., descending=True).column(...).item(0)
k_reason     = ...   # 1, 2 or 3 from the list above

print('k values :', k_values)
print('accuracy :', k_accuracies)
print('best k   :', best_k)

In [ ]:
grader.check("q3_4")

---

# **Part 4: Neighbourhoods like Ladywood**

The classifier answers a yes-or-no question. The distances underneath it answer
a more useful one.

If you are designing something for Ladywood, the places most like Ladywood are
where it might also work, and where you might look for what has already been
tried.

---

**Question 4.1.** Ladywood is 15 LSOAs. Build one average profile for the whole ward, then find the neighbourhoods most like it. **(12 Points)**

Assign:

1. `ladywood_profile` to an **array** holding the mean of each feature column across those 15 rows, in the same order as `all_features.labels`
2. `closest_ten` to a table of the ten neighbourhoods across Birmingham closest to that profile, with columns `LSOA`, `Ward` and `Distance`, sorted nearest first

*This is the first question where every line of the answer is blank.* The three lines at the top of the cell only prepare the tables; the two answers below them are yours to write from scratch, as are both questions in this part.

*Question 3.4 had its loop written out for you.* Here you write one, running over the feature columns instead of over values of k.

*Hint:* `tbl.apply` will not help with the first one. Build the array one column at a time: `make_array()` with no arguments gives you an empty array, and `np.append(array, item)` adds to it.

*Note:* `closest_ten` holds ten **LSOAs**, not ten wards, so the same ward will appear on more than one row. That is why the `LSOA` column is there: without it you could not tell those rows apart. Some of Ladywood's own LSOAs should be in the list, since they went into the average. Do not expect all fifteen, and do not assume Ladywood must come first: the average of fifteen places need not be closest to any one of them.

In [ ]:
all_features = lsoas.drop('LSOA', 'Ward', 'Households', 'No car %', 'Car status')
ladywood_rows = lsoas.where('Ward', are.equal_to('Ladywood'))
ladywood_features = ladywood_rows.drop('LSOA', 'Ward', 'Households', 'No car %', 'Car status')

ladywood_profile = ...
closest_ten = ...
closest_ten

In [ ]:
grader.check("q4_1")

---

**Question 4.2.** Look at the wards in your answer above. Which of the following is the most defensible thing to say about them? Assign `similarity_conclusion` to 1, 2 or 3. **(6 Points)**

1. These wards have the same problems as Ladywood, so the same solutions will work in all of them.
2. These wards resemble Ladywood on the measures in this dataset. Whether a design would transfer depends on much that this data does not capture.
3. These wards are the most deprived in Birmingham.

In [ ]:
similarity_conclusion = ...
similarity_conclusion

In [ ]:
grader.check("q4_2")

---

Run the cell below to see those ten on a map, together with every other
neighbourhood in the city. It plots the ten from **your** `closest_ten`, not a
fresh calculation, so the map shows what you actually produced.

In [ ]:
# Just run this cell
summary('q4_1')

The ten most similar neighbourhoods are not spread across the city. They sit in
a band through the inner ring, next to Ladywood itself.

Nothing in the calculation knew where any of these places were. It worked only from tenure,
housing type, deprivation and ethnicity, and it recovered geography anyway,
because those things and location are bound together.

---

## **Findings**

A classifier that predicts how car-free a neighbourhood is from its housing and
household composition, at well above the baseline you worked out in Question
1.1. That it works at all is the finding: **how people live and how they travel
are not separate problems.** Neighbourhoods of privately rented flats tend to be
neighbourhoods without cars, and a proposal that treats transport and housing
separately has missed something.

You also have a way of asking which places resemble which, which is how you find
precedents for a design rather than inventing from scratch.

What the data cannot tell you is why any of it is so, or what the people who
live there would want. That part is not in the census.

---

## **You're done!**

**Important submission information:**
- **Run all the tests** and verify that they all pass
- **Save** from the **File** menu
- **Run the last cell**, `grader.check_all()`, and check every question is listed
- **Click the download button to download the .ipynb file**
- **Then, go to Ulwazi and submit the .ipynb file to DDDM - Mini - Project 2 - Classification: Neighbourhoods Like Ladywood**

**It is your responsibility to make sure your work is saved before running the last cell.**

<img src="download_ipynb.png" alt="download ipynb" width="800"/>

In [ ]:
grader.check_all()

## **Submission**

Make sure you have run all cells in your notebook in order before clicking the download button to download the .ipynb file. **Please save your work before downloading!**